# Strava Activities — Data Preprocessing

This notebook reads the raw activity data exported by the fetch notebook, cleans and
transforms it, engineers analytical features, and exports the result to CSV and Excel.

**Pipeline:**
1. Load `strava_activities.json` into a Pandas DataFrame
2. Clean — flatten nested columns, fix types, audit nulls
3. Transform — convert API units to human-readable values
4. Feature engineering — temporal, performance, and categorical features
5. Export to CSV and Excel

### Section 1 — Libraries & Packages

All imports are declared here so every dependency is visible at a glance.

| Group | Packages | Purpose |
|---|---|---|
| Standard library | `pathlib`, `json` | File I/O and path handling |
| External | `pandas` | DataFrame operations and export |
| External | `numpy` | Vectorised operations and NaN handling |

In [ ]:
# Standard Library
from pathlib import Path
import json

# External Packages
import pandas as pd
import numpy as np

### Section 2 — Project Configuration

Defines the paths used throughout the notebook. The source JSON is produced by the
fetch notebook; the CSV and Excel paths are the export targets of this notebook.

| Constant | Description |
|---|---|
| `BASE_DIR` | Project root (current working directory) |
| `OUTPUT_DIR` | Directory for all exported files |
| `JSON_PATH` | Source file — raw activities from the fetch notebook |
| `CSV_PATH` | Export target — cleaned activities as CSV |
| `EXCEL_PATH` | Export target — cleaned activities as Excel |

In [ ]:
# Project root — all paths derived from here.
BASE_DIR = Path.cwd()

# ── Directory & file paths ────────────────────────────────────────────────────
OUTPUT_DIR = BASE_DIR / "output"

JSON_PATH  = OUTPUT_DIR / "strava_activities.json"  # Source (fetch notebook output)
CSV_PATH   = OUTPUT_DIR / "strava_activities.csv"   # Export target
EXCEL_PATH = OUTPUT_DIR / "strava_activities.xlsx"  # Export target

### Section 3 — Load JSON into DataFrame

`load_activities()` reads the JSON file and returns a raw Pandas DataFrame.
Each element of the top-level JSON array becomes one row (one activity).

In [ ]:
def load_activities(path=JSON_PATH):
    """
    Load the Strava activities JSON file into a Pandas DataFrame.

    Reads the file produced by ``export_to_json()`` in the fetch notebook.
    Each element of the top-level array becomes one row.

    Parameters
    ----------
    path : pathlib.Path or str, optional
        Path to the JSON file. Defaults to ``JSON_PATH``
        (``<project_root>/output/strava_activities.json``).

    Returns
    -------
    pandas.DataFrame
        Raw activities DataFrame as returned by the Strava API.

    Raises
    ------
    FileNotFoundError
        If the JSON file does not exist at ``path``.

    Usage
    -----
    df = load_activities()
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the fetch notebook first to generate it."
        )
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"Loaded {len(df)} activities with {df.shape[1]} columns.")
    return df

In [ ]:
df = load_activities()
df.head(2)

### Section 4 — Data Cleaning

Three focused functions applied in sequence. **No columns are dropped at this stage** —
all columns are preserved after flattening; column removal will be decided in a later
review pass after inspecting the full flattened structure.

1. `flatten_nested_columns(df)` — unpack dicts and lists into flat scalar columns
2. `parse_and_fix_types(df)` — parse datetimes, fix timezone strings, correct numeric types
3. `handle_nulls(df)` — audit and document null values

#### `flatten_nested_columns(df)` — Flatten Nested API Objects

The Strava API returns `athlete`, `map`, `start_latlng`, and `end_latlng` as nested
objects. This function extracts the useful scalar fields from each and drops the
original nested columns.

In [ ]:
def flatten_nested_columns(df):
    """
    Unpack nested dict and list columns into flat scalar columns.

    The Strava API returns several columns as nested objects stored as dicts or lists.
    This function extracts the useful scalar fields and drops the original nested columns,
    making the DataFrame fully flat and ready for type parsing and analysis.

    Transformations applied:
    - ``athlete``      dict → ``athlete_id``  (int)
    - ``map``          dict → ``has_route``   (bool: summary_polyline is non-empty)
    - ``start_latlng`` list → ``start_lat``,  ``start_lng`` (None for indoor activities)
    - ``end_latlng``   list → ``end_lat``,    ``end_lng``   (None for indoor activities)

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing raw nested columns from the Strava API.

    Returns
    -------
    pandas.DataFrame
        DataFrame with nested columns replaced by flat scalar equivalents.

    Usage
    -----
    df = flatten_nested_columns(df)
    """
    df = df.copy()

    # athlete dict → athlete_id (int)
    df["athlete_id"] = df["athlete"].apply(
        lambda x: x.get("id") if isinstance(x, dict) else None
    )
    df = df.drop(columns=["athlete"])

    # map dict → has_route (bool: True when a GPS polyline exists)
    df["has_route"] = df["map"].apply(
        lambda x: bool(x.get("summary_polyline")) if isinstance(x, dict) else False
    )
    df = df.drop(columns=["map"])

    # start_latlng list → start_lat, start_lng (None for empty lists / indoor activities)
    df["start_lat"] = df["start_latlng"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) == 2 else None
    )
    df["start_lng"] = df["start_latlng"].apply(
        lambda x: x[1] if isinstance(x, list) and len(x) == 2 else None
    )
    df = df.drop(columns=["start_latlng"])

    # end_latlng list → end_lat, end_lng
    df["end_lat"] = df["end_latlng"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) == 2 else None
    )
    df["end_lng"] = df["end_latlng"].apply(
        lambda x: x[1] if isinstance(x, list) and len(x) == 2 else None
    )
    df = df.drop(columns=["end_latlng"])

    print(f"After flattening: {df.shape[1]} columns.")
    return df

#### `parse_and_fix_types(df)` — Type Parsing & Corrections

Parses ISO 8601 datetime strings into timezone-aware Timestamps, converts
`utc_offset` from float seconds to integer hours, and strips escaped forward
slashes from timezone name strings.

In [ ]:
def parse_and_fix_types(df):
    """
    Parse datetime columns, fix timezone strings, and correct numeric types.

    Transformations applied:
    - ``start_date`` and ``start_date_local``: ISO 8601 string → timezone-aware datetime (UTC)
    - ``timezone``: strip escaped forward slashes left by some JSON serialisers
      (e.g. "Africa\/Addis_Ababa" → "Africa/Addis_Ababa")
    - ``utc_offset``: float seconds → int hours (10800.0 → 3)

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame

    Usage
    -----
    df = parse_and_fix_types(df)
    """
    df = df.copy()

    # Parse ISO 8601 datetime strings to tz-aware Timestamps (UTC)
    df["start_date"]       = pd.to_datetime(df["start_date"],       utc=True)
    df["start_date_local"] = pd.to_datetime(df["start_date_local"], utc=True)

    # Strip escaped forward slashes (safety: json.load handles this, but some serialisers miss it)
    df["timezone"] = df["timezone"].str.replace("\\/", "/", regex=False)

    # Convert utc_offset from float seconds to integer hours
    df["utc_offset"] = (df["utc_offset"] / 3600).astype(int)

    return df

#### `handle_nulls(df)` — Null Audit Checkpoint

Prints a summary of all null values. No imputation is applied — every null reflects
a genuine absence of data (e.g. indoor activities have no GPS or cadence data).

In [ ]:
def handle_nulls(df):
    """
    Audit null values and document the expected patterns. No imputation is applied.

    Null audit results:
    - ``average_heartrate``, ``max_heartrate``, ``suffer_score``:
      NaN for activities recorded without a heart rate sensor (2 of 46).
    - ``average_cadence``:
      NaN for indoor (trainer) workouts. Expected absence — no footpod or pedal sensor.
    - ``start_lat``, ``start_lng``, ``end_lat``, ``end_lng``:
      None for indoor activities. No GPS data by design.
    - ``location_city``, ``location_state``, ``location_country``:
      All null. Retained for the column review pass.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
        Unchanged (this function is a documented audit checkpoint).

    Usage
    -----
    df = handle_nulls(df)
    """
    df = df.copy()

    null_summary = df.isnull().sum()
    null_cols    = null_summary[null_summary > 0]

    print("Null value summary:")
    print(null_cols.to_string())

    return df

### Section 5 — Unit Conversions

The Strava API returns distances in metres, speeds in m/s, and times in seconds.
`convert_units()` creates human-readable equivalents and drops the raw originals.

| New column | Source | Formula |
|---|---|---|
| `distance_km` | `distance` (m) | ÷ 1000 |
| `moving_time_min` | `moving_time` (s) | ÷ 60 |
| `elapsed_time_min` | `elapsed_time` (s) | ÷ 60 |
| `average_speed_kmh` | `average_speed` (m/s) | × 3.6 |
| `max_speed_kmh` | `max_speed` (m/s) | × 3.6 |

Elevation columns (`elev_high`, `elev_low`, `total_elevation_gain`) remain in metres.

In [ ]:
def convert_units(df):
    """
    Convert Strava API raw units to human-readable equivalents.

    Creates new columns with familiar units, rounds them to a sensible precision,
    and drops the raw API originals.

    Conversions applied:
    - ``distance``      (m)   → ``distance_km``       (km)   = distance / 1000
    - ``moving_time``   (sec) → ``moving_time_min``   (min)  = moving_time / 60
    - ``elapsed_time``  (sec) → ``elapsed_time_min``  (min)  = elapsed_time / 60
    - ``average_speed`` (m/s) → ``average_speed_kmh`` (km/h) = average_speed × 3.6
    - ``max_speed``     (m/s) → ``max_speed_kmh``     (km/h) = max_speed × 3.6

    Elevation columns (``elev_high``, ``elev_low``, ``total_elevation_gain``) are left
    in metres as that is the standard unit for elevation.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame

    Usage
    -----
    df = convert_units(df)
    """
    df = df.copy()

    df["distance_km"]       = (df["distance"]      / 1000).round(3)
    df["moving_time_min"]   = (df["moving_time"]   / 60  ).round(2)
    df["elapsed_time_min"]  = (df["elapsed_time"]  / 60  ).round(2)
    df["average_speed_kmh"] = (df["average_speed"] * 3.6 ).round(3)
    df["max_speed_kmh"]     = (df["max_speed"]     * 3.6 ).round(3)

    df = df.drop(columns=["distance", "moving_time", "elapsed_time", "average_speed", "max_speed"])

    return df

### Section 6 — Feature Engineering

`engineer_features()` derives three groups of new analytical columns:

**Temporal** (from `start_date_local`): `date`, `year`, `month`, `month_name`, `day_of_week` (0=Sat), `day_name`, `hour_of_day`, `week_of_year`, `is_weekend`, `time_of_day`

**Performance**: `pace_min_per_km` (NaN for indoor), `is_outdoor`

**Categorical**: `distance_bucket` (Indoor / Short / Medium / Long), `hr_zone` (Zones 1–5 based on max HR 182 bpm)

In [ ]:
def engineer_features(df):
    """
    Derive analytical features from existing columns.

    Adds three groups of new columns without modifying the originals:

    **Temporal features** (derived from ``start_date_local``):
    - ``date``           — Date only (no time component)
    - ``year``           — Calendar year
    - ``month``          — Month number (1–12)
    - ``month_name``     — Month name (e.g. "January")
    - ``day_of_week``    — Day index: Saturday = 0, Sunday = 1, ..., Friday = 6
    - ``day_name``       — Day name (e.g. "Monday")
    - ``hour_of_day``    — Hour of the day (0–23)
    - ``week_of_year``   — ISO week number (1–53)
    - ``is_weekend``     — True for Saturday and Sunday
    - ``time_of_day``    — "Morning" (5–11h), "Afternoon" (12–17h),
                           "Evening" (18–21h), "Night" (22–4h)

    **Performance features**:
    - ``pace_min_per_km`` — Minutes per km. NaN for indoor activities (distance = 0 km).
    - ``is_outdoor``      — True when not a trainer workout and GPS coordinates exist.

    **Categorisation**:
    - ``distance_bucket`` — "Indoor" (0 km) | "Short" (<5 km) |
                            "Medium" (5–15 km) | "Long" (>15 km)
    - ``hr_zone``         — Heart rate zone based on max HR of 182 bpm. NaN if no HR data.
                            Zone 1: <109 | Zone 2: 109–127 | Zone 3: 128–147 |
                            Zone 4: 148–163 | Zone 5: ≥164

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain ``start_date_local``, ``moving_time_min``, ``distance_km``,
        ``trainer``, ``start_lat``, and ``average_heartrate``.

    Returns
    -------
    pandas.DataFrame

    Usage
    -----
    df = engineer_features(df)
    """
    df = df.copy()
    dt = df["start_date_local"].dt

    # ── Temporal features ─────────────────────────────────────────────────────
    df["date"]         = dt.date
    df["year"]         = dt.year
    df["month"]        = dt.month
    df["month_name"]   = dt.month_name()
    # Shift so Saturday = 0 (pandas default: Monday = 0, Saturday = 5)
    df["day_of_week"]  = (dt.dayofweek + 2) % 7
    df["day_name"]     = dt.day_name()
    df["hour_of_day"]  = dt.hour
    df["week_of_year"] = dt.isocalendar().week.astype(int)
    df["is_weekend"]   = df["day_of_week"].isin([0, 1])  # Saturday = 0, Sunday = 1

    def _time_of_day(hour):
        if 5 <= hour <= 11:
            return "Morning"
        elif 12 <= hour <= 17:
            return "Afternoon"
        elif 18 <= hour <= 21:
            return "Evening"
        else:
            return "Night"

    df["time_of_day"] = df["hour_of_day"].apply(_time_of_day)

    # ── Performance features ──────────────────────────────────────────────────
    df["pace_min_per_km"] = np.where(
        df["distance_km"] > 0,
        (df["moving_time_min"] / df["distance_km"]).round(2),
        np.nan
    )
    df["is_outdoor"] = ~df["trainer"] & df["start_lat"].notna()

    # ── Distance categorisation ───────────────────────────────────────────────
    conditions = [
        df["distance_km"] == 0,
        df["distance_km"] < 5,
        df["distance_km"] < 15,
    ]
    choices = ["Indoor", "Short", "Medium"]
    df["distance_bucket"] = np.select(conditions, choices, default="Long")

    # ── Heart rate zone (max HR = 182 bpm) ───────────────────────────────────
    hr_bins   = [0, 109, 128, 148, 164, float("inf")]
    hr_labels = ["Zone 1", "Zone 2", "Zone 3", "Zone 4", "Zone 5"]
    df["hr_zone"] = pd.cut(
        df["average_heartrate"],
        bins=hr_bins,
        labels=hr_labels,
        right=False
    )

    print(f"Feature engineering complete. DataFrame now has {df.shape[1]} columns.")
    return df

### Section 7 — Preview Cleaned DataFrame

Run the full pipeline up to this point and inspect the result before exporting.

In [ ]:
print(f'Shape: {df.shape}')
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.head()

### Section 8 — Export

Two functions that write the preprocessed DataFrame to disk:
- `export_to_csv()` — CSV with UTF-8 BOM so Excel opens it without encoding issues
- `export_to_excel()` — Excel workbook via `openpyxl`

In [ ]:
def export_to_csv(df, path=CSV_PATH):
    """
    Export the activities DataFrame to a CSV file.

    Uses ``utf-8-sig`` encoding (UTF-8 with BOM) so Excel opens the file correctly
    without character encoding issues on any platform.

    Parameters
    ----------
    df : pandas.DataFrame
        The preprocessed activities DataFrame.
    path : pathlib.Path or str, optional
        Destination file path. Defaults to ``CSV_PATH``
        (``<project_root>/output/strava_activities.csv``).

    Returns
    -------
    None
        Writes the file and prints a confirmation message.

    Usage
    -----
    export_to_csv(df)
    export_to_csv(df, path="my_folder/activities.csv")
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Exported {len(df)} activities to {path}")

In [ ]:
def export_to_excel(df, path=EXCEL_PATH):
    """
    Export the activities DataFrame to an Excel workbook.

    Writes a single worksheet named "Strava Activities" using the openpyxl engine.
    Timezone-aware datetime columns are converted to timezone-naive before writing
    because Excel does not support timezone-aware timestamps.

    Parameters
    ----------
    df : pandas.DataFrame
        The preprocessed activities DataFrame.
    path : pathlib.Path or str, optional
        Destination file path. Defaults to ``EXCEL_PATH``
        (``<project_root>/output/strava_activities.xlsx``).

    Returns
    -------
    None
        Writes the file and prints a confirmation message.

    Usage
    -----
    export_to_excel(df)
    export_to_excel(df, path="my_folder/activities.xlsx")
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Excel does not support tz-aware datetimes — strip timezone info before writing.
    df_excel = df.copy()
    for col in df_excel.select_dtypes(include=["datetimetz"]).columns:
        df_excel[col] = df_excel[col].dt.tz_localize(None)

    df_excel.to_excel(path, index=False, sheet_name="Strava Activities", engine="openpyxl")
    print(f"Exported {len(df)} activities to {path}")

### Pipeline Orchestration

`run_pipeline()` chains all steps in order and returns the final DataFrame.
Run this single cell to execute the complete preprocessing workflow end-to-end.

In [ ]:
def run_pipeline():
    """
    Run the full preprocessing pipeline end-to-end.

    Chains all preprocessing steps in order:
    1. Load JSON → DataFrame
    2. Flatten nested columns
    3. Parse types & fix strings
    4. Audit nulls
    5. Convert units
    6. Engineer features
    7. Export to CSV and Excel

    Returns
    -------
    pandas.DataFrame
        Fully cleaned, transformed, and feature-engineered activities DataFrame.

    Usage
    -----
    df = run_pipeline()
    """
    df = load_activities()
    df = flatten_nested_columns(df)
    df = parse_and_fix_types(df)
    df = handle_nulls(df)
    df = convert_units(df)
    df = engineer_features(df)
    export_to_csv(df)
    export_to_excel(df)
    return df


df = run_pipeline()
df